# Smart Trip Budget Planner | Single Agent (ReAct)

In [1]:
import ast
import operator
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
# Web search tool (free, no API key; requires `duckduckgo-search` package)
search_tool = DuckDuckGoSearchResults(name="web_search")

In [4]:
# Safe arithmetic evaluator (no eval() -- prevents code injection)
SAFE_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

In [5]:
def safe_eval(expr: str):
    """Safely evaluate arithmetic expressions only."""
    tree = ast.parse(expr, mode="eval")
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        elif isinstance(node, ast.BinOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](_eval(node.operand))
        else:
            raise ValueError(f"Unsupported expression: {ast.dump(node)}")
    return _eval(tree)

In [6]:
# Currency converter with static demo exchange rates
EXCHANGE_RATES = {
    ("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09,
    ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
    ("USD", "JPY"): 149.50, ("JPY", "USD"): 0.0067,
    ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16,
    ("EUR", "JPY"): 162.70, ("JPY", "EUR"): 0.0061,
    ("GBP", "JPY"): 189.40, ("JPY", "GBP"): 0.0053,
}

In [7]:
# Tools

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '150 * 5 + 80 * 3'"""
    try:
        result = safe_eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies. Supported: USD, EUR, GBP, JPY."""
    from_c = from_currency.upper()
    to_c = to_currency.upper()
    if from_c == to_c:
        return f"{amount:.2f} {from_c} = {amount:.2f} {to_c}"
    rate = EXCHANGE_RATES.get((from_c, to_c))
    if not rate:
        return f"Conversion from {from_c} to {to_c} is not supported."
    converted = amount * rate
    return f"{amount:.2f} {from_c} = {converted:.2f} {to_c} (rate: {rate})"

In [8]:
# Create the agent
model = ChatOpenAI(model="gpt-4o")
agent = create_agent(
    model=model,
    tools=[search_tool, calculator, currency_converter],
    system_prompt=(
        "You are a smart trip budget planner. Help users estimate travel costs. "
        "Search the web for prices (flights, hotels, food), use the calculator for totals, "
        "and convert currencies as needed. Always provide a breakdown and final total."
    )
)

In [9]:
# Plot the agent
plot_mermaid(agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
# Run the agent
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport."
    }]
})

print(result["messages"][-1].content)

Here's a breakdown of the estimated costs for a 5-day trip from New York to Tokyo:

1. **Flights:**
   - Round-trip flights from New York to Tokyo in December 2023 are approximately $1,150.

2. **Accommodation:**
   - The average nightly cost for a hotel in Tokyo in December 2023 ranges around $150, resulting in a total of $750 for 5 nights.

3. **Food:**
   - The average daily cost for food in Tokyo is about ¥4,526, which is approximately $32 USD per day (using a rough exchange rate of 1 USD = 140 JPY). For 5 days, this totals to $160.

4. **Transport:**
   - The average daily transport cost in Tokyo is around ¥1,000, approximately $7 USD per day. For 5 days, this totals to $35.

### Total Estimated Budget:
- **Flights:** $1,150
- **Hotels:** $750
- **Food:** $160
- **Transport:** $35

#### **Grand Total:** $2,095

Please note that these are estimates and actual prices can vary due to changes in exchange rates, availability, and specific travel dates. Consider booking in advance for m

In [11]:
# Streaming

stream_invoke(
    agent, {
        "messages": [{
            "role": "user",
            "content": "Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport."
        }]
    })

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

┌─ HUMAN

│ 

...

└────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'messages': [HumanMessage(content='Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport.', additional_kwargs={}, response_metadata={}, id='d2746f67-b45b-4bed-96b7-b00ba156f283'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 214, 'total_tokens': 320, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ab062d0c27', 'id': 'chatcmpl-DRNueX0yl63zFqwoJMm5bL9E2HMoK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5f3a-44dd-7342-9120-c48777eb11ec-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'round trip flight cost New York to Tokyo December 2023'}, 'id': 'call_W7hNd